# SHWD Stage 2 Ablation Setup

This notebook initializes Stage 2 (Custom Module Ablation Study) for Safety Helmet Detection.
It automatically detects Kaggle input datasets & notebook outputs, extracts Top-2 weights (`yolo11s_best.pt`, `yolov8s_best.pt`), converts VOC2028 to YOLO format, and smoke-tests custom modules (`CoordConv`, `RepConv`, `BiFormer`, `Focal-EIoU`).

**Mounted Kaggle Inputs Handled:**
- Dataset: `VOC2028` (`/kaggle/input/datasets/hannhu4002/voc2028`)
- Dataset: `shwd-benchmark-code` (`/kaggle/input/datasets/hannhu4002/shwd-benchmark-code`)
- Notebook Output: `Structural Re-parameterized YOLO Architecture1`
- Notebook Output: `Structural Re-parameterized YOLO Architecture2`
- Notebook Output: `SHWD_Baseline_Consolidated_2` (contains `SHWD_Compact_Outputs.zip`)

## Stage 2 Execution Settings

This block is intentionally similar to the Stage 1 notebook. For normal use, run setup first, then toggle the ablation runs you want.

Important distinction:
- Stage 1 used stock Ultralytics models, so `RUN_FULL_TRAINING=True` could safely launch all baselines.
- Stage 2 changes architecture/loss. A0 is immediately runnable. A1 can be run as an Albumentations offline-augmented dataset experiment. A2-A6 are controlled custom-module experiments and must be integrated carefully before full 100-epoch runs.


In [1]:
# Step 0: Stage 2 Settings / Toggles
from pathlib import Path

# Master switch. Keep False while validating each ablation. When True, cells use STAGE2_FULL_EPOCHS.
RUN_STAGE2_FULL = True

# Setup checks
RUN_SETUP_CONVERT = True
RUN_MODULE_SMOKE_TEST = True

# Runnable Stage 2 experiments in this notebook
RUN_A0_CONTROL_EVAL = False          # Re-evaluate yolo11s_best.pt and yolov8s_best.pt as controls.
RUN_A1_BUILD_AUG_DATASET = True    # Build offline Albumentations hard-case train set.
RUN_A1_HARDCASE_AUG_TRAIN = False   # Fine-tune top-2 weights on the A1 augmented dataset.

# Stage 2 custom-module switches. These are intentionally guarded until a custom trainer/model YAML is patched.
RUN_A2_COORDCONV = False
RUN_A3_REPCONV_REPC3 = False
RUN_A4_FOCAL_EIOU_ALPHA_FOCAL = True
RUN_A5_BIFORMER = False
RUN_A6_FULL_FUSION = False

# Training settings
STAGE2_SMOKE_EPOCHS = 3
STAGE2_FULL_EPOCHS = 100
STAGE2_IMGSZ = 640
STAGE2_BATCH = 16
STAGE2_DEVICE = '0,1'       # train on Dual T4
STAGE2_EVAL_DEVICE = '0'    # val/predict/export on one GPU to avoid DDP predict issues
STAGE2_WORKERS = 4
STAGE2_PATIENCE = 50
STAGE2_SEED = 3407

# A1 hard-case offline augmentation controls. None means all train images; use a small number for smoke tests.
A1_AUGMENT_LIMIT = 256 if not RUN_STAGE2_FULL else None
A1_JPEG_QUALITY = 92

# Top-2 selected after Stage 1
STAGE2_BACKBONES = ['yolo11s', 'yolov8s']

print('RUN_STAGE2_FULL =', RUN_STAGE2_FULL)
print('Runnable now: A0 control eval =', RUN_A0_CONTROL_EVAL, '| A1 build/train =', RUN_A1_BUILD_AUG_DATASET, RUN_A1_HARDCASE_AUG_TRAIN)
print('Custom guarded toggles: A2=', RUN_A2_COORDCONV, 'A3=', RUN_A3_REPCONV_REPC3, 'A4=', RUN_A4_FOCAL_EIOU_ALPHA_FOCAL, 'A5=', RUN_A5_BIFORMER, 'A6=', RUN_A6_FULL_FUSION)
print('Epochs selected =', STAGE2_FULL_EPOCHS if RUN_STAGE2_FULL else STAGE2_SMOKE_EPOCHS)


RUN_STAGE2_FULL = True
Runnable now: A0 control eval = False | A1 build/train = True False
Custom guarded toggles: A2= False A3= False A4= True A5= False A6= False
Epochs selected = 100


In [2]:
# Step 1: Install Dependencies
from pathlib import Path
import sys
import subprocess

def find_file(name):
    candidates = [Path.cwd() / name, Path('/kaggle/working') / name]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(name))
    return next((p for p in candidates if p.exists()), None)

req_path = find_file('requirements_kaggle.txt')
if req_path:
    print('Installing from:', req_path)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', '-r', str(req_path)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'onnx', 'onnxruntime-gpu', 'pandas', 'albumentations', 'opencv-python-headless'], check=True)

Installing from: /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/requirements_kaggle.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 30.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.

In [3]:
# Step 2: Automatic Flexible Path & Weight Resolution
from pathlib import Path
import os
import shutil
import zipfile
import subprocess
import sys

def find_dir_by_markers(dirname, required_children):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob(dirname):
            if p.is_dir() and all((p / child).exists() for child in required_children):
                return p
    return None

def find_file_anywhere(name):
    roots = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle/input')]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.exists():
            return direct
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None

# 1. Locate VOC2028 Dataset Root
DATASET_ROOT = find_dir_by_markers('VOC2028', ['Annotations', 'JPEGImages', 'ImageSets'])
print('📌 DATASET_ROOT =', DATASET_ROOT)

# 2. Resolve Compact Artifacts & Top-2 Model Weights
EXTRACT_DIR = Path('/kaggle/working/extracted_compact')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Look for SHWD_Compact_Outputs.zip inside SHWD_Baseline_Consolidated_2
compact_zip = find_file_anywhere('SHWD_Compact_Outputs.zip')
if compact_zip and compact_zip.exists():
    print(f'📦 Found Compact ZIP at: {compact_zip}. Extracting...')
    with zipfile.ZipFile(compact_zip, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f'✅ Extracted artifacts to: {EXTRACT_DIR}')

# Locate Compact Root (either extracted zip or directory)
COMPACT_ROOT = find_dir_by_markers('SHWD_Compact_Outputs', ['weights', 'master_benchmark_results.csv'])
if COMPACT_ROOT is None and (EXTRACT_DIR / 'master_benchmark_results.csv').exists():
    COMPACT_ROOT = EXTRACT_DIR
print('📌 COMPACT_ROOT =', COMPACT_ROOT)

# 3. Resolve Support Scripts
SCRIPT_PATH = find_file_anywhere('kaggle_shwd_baseline.py')
MODULE_PATH = find_file_anywhere('custom_ablation_modules.py')
AUG_PATH = find_file_anywhere('albumentations_hardcase_policy.py')

print('📌 SCRIPT_PATH =', SCRIPT_PATH)
print('📌 MODULE_PATH =', MODULE_PATH)
print('📌 AUG_PATH =', AUG_PATH)

# Verify code scripts & dataset exist
if not DATASET_ROOT or not SCRIPT_PATH or not MODULE_PATH:
    raise FileNotFoundError('Missing required VOC2028 dataset or python scripts! Check input attachments.')

# 4. Flexible Top-2 Model Weights Lookup (Zip Extraction or Raw Run Outputs)
TOP2_WEIGHTS = {}

# Try 1: Look in COMPACT_ROOT weights
if COMPACT_ROOT and (COMPACT_ROOT / 'weights').exists():
    w_dir = COMPACT_ROOT / 'weights'
    y11 = list(w_dir.glob('*yolo11s*best*.pt'))
    y8 = list(w_dir.glob('*yolov8s*best*.pt'))
    if y11:
        TOP2_WEIGHTS['yolo11s'] = y11[0]
    if y8:
        TOP2_WEIGHTS['yolov8s'] = y8[0]

# Try 2: Direct lookup in raw mounted notebook outputs if zip didn't have them
if 'yolo11s' not in TOP2_WEIGHTS or not TOP2_WEIGHTS['yolo11s'].exists():
    for p in Path('/kaggle/input').rglob('best.pt'):
        p_str = str(p).lower()
        if 'yolo11s' in p_str and ('architecture2' in p_str or 'architecture-2' in p_str):
            TOP2_WEIGHTS['yolo11s'] = p
            break

if 'yolov8s' not in TOP2_WEIGHTS or not TOP2_WEIGHTS['yolov8s'].exists():
    for p in Path('/kaggle/input').rglob('best.pt'):
        p_str = str(p).lower()
        if 'yolov8s' in p_str and ('architecture1' in p_str or 'architecture-1' in p_str):
            TOP2_WEIGHTS['yolov8s'] = p
            break

# Print weights status
print('\n--- Top-2 Champion Weights Found ---')
for name, path in TOP2_WEIGHTS.items():
    mb = round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else 0
    print(f'  - {name}: {path} (exists={path.exists()}, size={mb} MB)')
    if not path.exists():
        raise FileNotFoundError(f'Missing weights for {name}: {path}')

WORK_ROOT = Path('/kaggle/working/SHWD_STAGE2')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('\n✅ Stage 2 Setup Initialized Successfully at WORK_ROOT =', WORK_ROOT)

📌 DATASET_ROOT = /kaggle/input/datasets/hannhu4002/voc2028/VOC2028
📦 Found Compact ZIP at: /kaggle/input/notebooks/hannhu4002/shwd-baseline-consolidated-2/SHWD_Compact_Outputs.zip. Extracting...
✅ Extracted artifacts to: /kaggle/working/extracted_compact
📌 COMPACT_ROOT = /kaggle/working/extracted_compact
📌 SCRIPT_PATH = /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/kaggle_shwd_baseline.py
📌 MODULE_PATH = /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/custom_ablation_modules.py
📌 AUG_PATH = /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/albumentations_hardcase_policy.py

--- Top-2 Champion Weights Found ---
  - yolo11s: /kaggle/working/extracted_compact/weights/yolo11s_best.pt (exists=True, size=18.28 MB)
  - yolov8s: /kaggle/working/extracted_compact/weights/yolov8s_best.pt (exists=True, size=21.46 MB)

✅ Stage 2 Setup Initialized Successfully at WORK_ROOT = /kaggle/working/SHWD_STAGE2


In [4]:
# Step 3: Convert VOC2028 to YOLO format for Stage 2.
OUTPUT_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2')
if RUN_SETUP_CONVERT:
    cmd = [
        sys.executable, str(SCRIPT_PATH),
        '--mode', 'convert',
        '--dataset-root', str(DATASET_ROOT),
        '--output-dir', str(OUTPUT_DIR),
        '--overwrite',
    ]
    print('Executing command:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping conversion because RUN_SETUP_CONVERT=False. Expected existing:', OUTPUT_DIR)


Executing command: /usr/bin/python3 /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/kaggle_shwd_baseline.py --mode convert --dataset-root /kaggle/input/datasets/hannhu4002/voc2028/VOC2028 --output-dir /kaggle/working/SHWD_YOLO_STAGE2 --overwrite
{
  "dataset_root": "/kaggle/input/datasets/hannhu4002/voc2028/VOC2028",
  "output_dir": "/kaggle/working/SHWD_YOLO_STAGE2",
  "train_images": 6064,
  "test_images": 1517,
  "labels_written": 7581,
  "missing_images": 0,
  "missing_xml": 0,
  "invalid_boxes": 0,
  "ignored_objects": 3,
  "class_counts": {
    "hat": 9044,
    "person": 111514
  },
  "ignored_by_class": {
    "dog": 3
  },
  "dry_run": false
}


In [5]:
# Step 4: Verify Conversion Report
import json

report_path = OUTPUT_DIR / 'conversion_report.json'
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
assert report['train_images'] == 6064, report['train_images']
assert report['test_images'] == 1517, report['test_images']
assert report['class_counts']['hat'] == 9044, report['class_counts']
assert report['class_counts']['person'] == 111514, report['class_counts']
assert report['ignored_by_class'].get('dog') == 3, report['ignored_by_class']
print('✅ Stage 2 dataset conversion verified successfully!')

{
  "dataset_root": "/kaggle/input/datasets/hannhu4002/voc2028/VOC2028",
  "output_dir": "/kaggle/working/SHWD_YOLO_STAGE2",
  "train_images": 6064,
  "test_images": 1517,
  "labels_written": 7581,
  "missing_images": 0,
  "missing_xml": 0,
  "invalid_boxes": 0,
  "ignored_objects": 3,
  "class_counts": {
    "hat": 9044,
    "person": 111514
  },
  "ignored_by_class": {
    "dog": 3
  },
  "dry_run": false
}
✅ Stage 2 dataset conversion verified successfully!


In [6]:
# Step 5: Copy Top-2 Weights to Working Directory
STAGE2_WEIGHTS = WORK_ROOT / 'weights'
STAGE2_WEIGHTS.mkdir(parents=True, exist_ok=True)
for name, src in TOP2_WEIGHTS.items():
    dst = STAGE2_WEIGHTS / src.name
    shutil.copy2(src, dst)
    print(f'Copied {name} -> {dst} ({round(dst.stat().st_size / (1024 * 1024), 2)} MB)')

Copied yolo11s -> /kaggle/working/SHWD_STAGE2/weights/yolo11s_best.pt (18.28 MB)
Copied yolov8s -> /kaggle/working/SHWD_STAGE2/weights/yolov8s_best.pt (21.46 MB)


In [7]:
# Step 6: Smoke-Test Custom Modules under Kaggle PyTorch
if RUN_MODULE_SMOKE_TEST:
    cmd = [sys.executable, str(MODULE_PATH)]
    print('Running module test:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping custom module smoke test because RUN_MODULE_SMOKE_TEST=False')


Running module test: /usr/bin/python3 /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/custom_ablation_modules.py
{'shape': (2, 24, 32, 32), 'repconv_fusion_max_diff': 2.1457672119140625e-06}
{'focal_eiou': 0.20399829745292664}


In [8]:
# Step 7: Display Master Benchmark Leaderboard
import pandas as pd

csv_path = find_file_anywhere('master_benchmark_results.csv')
if csv_path and csv_path.exists():
    df = pd.read_csv(csv_path)
    cols = [c for c in ['model', 'map50', 'map50_95', 'ap50_hat', 'ap_hat', 'recall_hat', 'precision_hat', 'f1_hat', 'speed_inference_ms', 'onnx_latency_mean_ms', 'onnx_fps_mean', 'best_pt_mb'] if c in df.columns]
    print('🏆 Master Stage 1 Baseline Matrix:')
    display(df[cols].sort_values(['map50_95', 'map50'], ascending=False))
else:
    print('⚠️ Master CSV not found, setup ready for Stage 2 ablation training.')

🏆 Master Stage 1 Baseline Matrix:


,model,map50,map50_95,ap50_hat,ap_hat,recall_hat,precision_hat,f1_hat,speed_inference_ms,onnx_latency_mean_ms,onnx_fps_mean,best_pt_mb
0,yolo11s.pt,0.947379,0.625415,0.940598,0.742646,0.903456,0.911165,0.907294,6.524133,144.5346,6.92,18.278
1,yolov8s.pt,0.948927,0.622085,0.942811,0.737160,0.906199,0.917859,0.911991,6.105510,187.6546,5.33,21.465
2,yolov10s.pt,0.943913,0.621872,0.933565,0.734777,0.891388,0.918822,0.904897,6.227032,147.7368,6.77,15.749
3,yolov10n.pt,0.932999,0.603535,0.929662,0.718302,0.871340,0.915824,0.893028,2.663596,90.6514,11.03,5.475
4,yolov8n.pt,0.932052,0.602583,0.924177,0.718098,0.871640,0.904821,0.887921,2.855711,63.4755,15.75,5.949
5,yolo11n.pt,0.931852,0.602088,0.923260,0.715203,0.865058,0.930009,0.896358,3.182678,68.2158,14.66,5.207


## Next Execution Step: Stage 2 Custom Ablation Experiments

After this setup notebook passes, begin custom ablation training using **`yolo11s_best.pt`** and **`yolov8s_best.pt`** as controls.

**Ablation Progression:**
1. **A0**: Re-evaluate `yolo11s_best.pt` & `yolov8s_best.pt` as control baselines.
2. **A1**: Apply hard-case online augmentation (Albumentations: RandomShadow, HSV shift, Cutout).
3. **A2**: Inject **CoordConv** in stem/neck.
4. **A3**: Inject **RepConv / RepC3** and verify `switch_to_deploy()` fusion.
5. **A4**: Apply **Focal-EIoU Loss** for bounding box regression + alpha-focal classification for `hat`.
6. **A5**: Inject **BiFormer Attention** if small/occluded helmets require fine-grained routing.
7. **A6**: Train final combined candidate.

## Stage 2 Runner Cells

The following cells make the setup notebook usable like Stage 1: you can toggle A0/A1 from the settings block. A2-A6 remain guarded because they require patching architecture/loss into a custom trainer, not just changing a stock Ultralytics training flag.


In [9]:
# Step 8: A0 Control Evaluation
# Re-evaluate the selected top-2 weights on the Stage 2 YOLO dataset.
if RUN_A0_CONTROL_EVAL:
    from ultralytics import YOLO
    import csv
    import json

    a0_dir = WORK_ROOT / 'A0_control_eval'
    a0_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for name in STAGE2_BACKBONES:
        weight = TOP2_WEIGHTS[name]
        print(f'\n[A0] Evaluating {name}: {weight}')
        model = YOLO(str(weight))
        metrics = model.val(
            data=str(OUTPUT_DIR / 'shwd.yaml'),
            imgsz=STAGE2_IMGSZ,
            device=STAGE2_EVAL_DEVICE,
            split='val',
            plots=True,
            project=str(a0_dir),
            name=name,
            exist_ok=True,
        )

        row = {
            'ablation_id': 'A0_control',
            'model': name,
            'weight': str(weight),
            'map50': getattr(metrics.box, 'map50', None),
            'map50_95': getattr(metrics.box, 'map', None),
            'map75': getattr(metrics.box, 'map75', None),
        }
        maps = getattr(metrics.box, 'maps', None)
        if maps is not None and len(maps) >= 2:
            row['ap_hat'] = float(maps[0])
            row['ap_person'] = float(maps[1])
        rows.append(row)
        print(row)

    out_csv = a0_dir / 'A0_control_results.csv'
    with out_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r.keys()}))
        writer.writeheader()
        writer.writerows(rows)
    print('Saved:', out_csv)
else:
    print('Skipping A0 because RUN_A0_CONTROL_EVAL=False and RUN_STAGE2_FULL=False')


Skipping A0 because RUN_A0_CONTROL_EVAL=False and RUN_STAGE2_FULL=False


In [10]:
# Step 9: A1 Build Offline Albumentations Hard-Case Dataset
# This creates a train set containing original train images plus augmented copies.
# It is disabled by default because full offline augmentation writes many images to /kaggle/working.
if RUN_A1_BUILD_AUG_DATASET:
    import cv2
    import shutil
    import sys
    from tqdm.auto import tqdm

    module_dir = Path(AUG_PATH).parent
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))
    from albumentations_hardcase_policy import augment_one_yolo_sample

    A1_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG')
    for sub in ['images/train', 'labels/train', 'images/test', 'labels/test']:
        (A1_DIR / sub).mkdir(parents=True, exist_ok=True)

    def safe_link_or_copy(src, dst):
        dst = Path(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() or dst.is_symlink():
            return
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)

    # Link/copy original train and test files.
    for split in ['train', 'test']:
        for img in (OUTPUT_DIR / 'images' / split).glob('*'):
            safe_link_or_copy(img, A1_DIR / 'images' / split / img.name)
        for lab in (OUTPUT_DIR / 'labels' / split).glob('*.txt'):
            shutil.copy2(lab, A1_DIR / 'labels' / split / lab.name)

    train_images = sorted((OUTPUT_DIR / 'images/train').glob('*'))
    if A1_AUGMENT_LIMIT is not None:
        train_images = train_images[:A1_AUGMENT_LIMIT]
    print(f'[A1] Building augmented copies for {len(train_images)} train images')

    ok = 0
    for img in tqdm(train_images):
        stem = img.stem
        label = OUTPUT_DIR / 'labels/train' / f'{stem}.txt'
        out_img = A1_DIR / 'images/train' / f'{stem}_hardaug.jpg'
        out_lab = A1_DIR / 'labels/train' / f'{stem}_hardaug.txt'
        if augment_one_yolo_sample(img, label, out_img, out_lab, image_size=STAGE2_IMGSZ):
            ok += 1
    print(f'[A1] Augmented images written: {ok}')

    yaml_text = '\n'.join([
        f'path: {A1_DIR.as_posix()}',
        'train: images/train',
        'val: images/test',
        'test: images/test',
        'nc: 2',
        'names:',
        '  0: hat',
        '  1: person',
        '',
    ])
    (A1_DIR / 'shwd_a1_hardaug.yaml').write_text(yaml_text, encoding='utf-8')
    print('A1 YAML:', A1_DIR / 'shwd_a1_hardaug.yaml')
else:
    print('Skipping A1 dataset build because RUN_A1_BUILD_AUG_DATASET=False')


[A1] Building augmented copies for 6064 train images


  0%|          | 0/6064 [00:00<?, ?it/s]

[A1] Augmented images written: 6064
A1 YAML: /kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG/shwd_a1_hardaug.yaml


In [11]:
# Step 10: A1 Hard-Case Augmentation Fine-Tuning
# Starts from yolo11s_best.pt and yolov8s_best.pt, then fine-tunes on the A1 augmented dataset.
if RUN_A1_HARDCASE_AUG_TRAIN:
    from ultralytics import YOLO

    A1_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG')
    A1_YAML = A1_DIR / 'shwd_a1_hardaug.yaml'
    if not A1_YAML.exists():
        raise FileNotFoundError('Run Step 9 with RUN_A1_BUILD_AUG_DATASET=True before A1 training.')

    epochs = STAGE2_FULL_EPOCHS if RUN_STAGE2_FULL else STAGE2_SMOKE_EPOCHS
    for name in STAGE2_BACKBONES:
        weight = TOP2_WEIGHTS[name]
        print(f'\n[A1] Fine-tuning {name} for {epochs} epochs from {weight}')
        model = YOLO(str(weight))
        model.train(
            data=str(A1_YAML),
            epochs=epochs,
            imgsz=STAGE2_IMGSZ,
            batch=STAGE2_BATCH,
            device=STAGE2_DEVICE,
            workers=STAGE2_WORKERS,
            patience=STAGE2_PATIENCE,
            seed=STAGE2_SEED,
            deterministic=True,
            project=str(WORK_ROOT / 'A1_hardcase_aug_train'),
            name=name,
            exist_ok=True,
            save=True,
            save_period=-1,
            cache=False,
            plots=True,
            hsv_h=0.025,
            hsv_s=0.75,
            hsv_v=0.45,
            translate=0.10,
            scale=0.50,
            fliplr=0.50,
            mosaic=1.00,
            mixup=0.10,
            close_mosaic=10,
        )
else:
    print('Skipping A1 training because RUN_A1_HARDCASE_AUG_TRAIN=False and RUN_STAGE2_FULL=False')


Skipping A1 training because RUN_A1_HARDCASE_AUG_TRAIN=False and RUN_STAGE2_FULL=False


## Step 11: A2 CoordConv Fine-Tuning

This cell fine-tunes the Top-2 backbones (`yolo11s`, `yolov8s`) by replacing the standard Conv stem with a **`CoordConv`** module.
It appends normalized spatial $x$ and $y$ coordinate channels to the input image tensor before convolution, allowing the model to distinguish ground-level distractors (e.g. yellow buckets, traffic cones) from helmets worn on heads.

In [12]:
# Step 11: A2 CoordConv Fine-Tuning
if RUN_A2_COORDCONV:
    from ultralytics import YOLO
    import sys
    import os
    import site
    from pathlib import Path
    import torch
    import torch.nn as nn

    # Ensure custom_ablation_modules is importable
    module_dir = Path(MODULE_PATH).parent
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))
    from custom_ablation_modules import CoordConv

    # Write ablation_trainer.py into site-packages and working dir so PyTorch DDP subprocesses can import it natively on all GPUs
    trainer_code = (
        'from ultralytics.models.yolo.detect import DetectionTrainer\n\n'
        'class AblationTrainer(DetectionTrainer):\n'
        '    def get_model(self, cfg=None, weights=None, verbose=True):\n'
        '        return getattr(self, "patched_model", super().get_model(cfg, weights, verbose))\n'
    )

    # 1. Write to /kaggle/working/ablation_trainer.py
    work_dir = '/kaggle/working'
    (Path(work_dir) / 'ablation_trainer.py').write_text(trainer_code, encoding='utf-8')
    if work_dir not in sys.path:
        sys.path.insert(0, work_dir)

    # 2. Write to Python site-packages (globally importable by all DDP rank subprocesses)
    for sp in site.getsitepackages():
        try:
            (Path(sp) / 'ablation_trainer.py').write_text(trainer_code, encoding='utf-8')
        except Exception:
            pass

    # 3. Export PYTHONPATH to environment for DDP launcher
    existing_pp = os.environ.get('PYTHONPATH', '')
    if work_dir not in existing_pp:
        os.environ['PYTHONPATH'] = f"{work_dir}:{existing_pp}" if existing_pp else work_dir

    from ablation_trainer import AblationTrainer

    # Check dataset: Prefer A1 augmented dataset to preserve cumulative progression (Data Aug + CoordConv)
    A1_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG')
    A1_YAML = A1_DIR / 'shwd_a1_hardaug.yaml'
    if A1_YAML.exists():
        data_yaml = A1_YAML
        print(f'[A2 CoordConv] Cumulative Mode: Training on A1 Hard-Case Augmented dataset ({data_yaml})')
    else:
        data_yaml = OUTPUT_DIR / 'shwd.yaml'
        print(f'[A2 CoordConv] Standard Mode: Training on Stage 2 dataset ({data_yaml})')

    epochs = STAGE2_FULL_EPOCHS if RUN_STAGE2_FULL else STAGE2_SMOKE_EPOCHS

    for name in STAGE2_BACKBONES:
        weight = TOP2_WEIGHTS[name]
        print(f'\n[A2 CoordConv] Patching stem & fine-tuning {name} for {epochs} epochs from {weight} on Dual GPU ({STAGE2_DEVICE})')
        base_model = YOLO(str(weight))

        # Patch Stem layer 0 with CoordConv
        stem = base_model.model.model[0]
        c1 = 3
        c2 = stem.conv.out_channels
        k = stem.conv.kernel_size[0] if hasattr(stem.conv, 'kernel_size') else 3
        s = stem.conv.stride[0] if hasattr(stem.conv, 'stride') else 2

        coord_stem = CoordConv(c1=c1, c2=c2, k=k, s=s)
        coord_stem.i = getattr(stem, 'i', 0)
        coord_stem.f = getattr(stem, 'f', -1)
        coord_stem.type = 'CoordConv'
        base_model.model.model[0] = coord_stem

        # Configure AblationTrainer overrides for Dual GPU DDP
        overrides = {
            'model': str(weight),
            'data': str(data_yaml),
            'epochs': epochs,
            'imgsz': STAGE2_IMGSZ,
            'batch': STAGE2_BATCH,
            'device': STAGE2_DEVICE,
            'workers': STAGE2_WORKERS,
            'patience': STAGE2_PATIENCE,
            'seed': STAGE2_SEED,
            'deterministic': True,
            'project': str(WORK_ROOT / 'A2_coordconv_train'),
            'name': name,
            'exist_ok': True,
            'save': True,
            'save_period': -1,
            'cache': False,
            'plots': True,
        }

        trainer = AblationTrainer(overrides=overrides)
        trainer.patched_model = base_model.model
        trainer.train()
else:
    print('Skipping A2 CoordConv training because RUN_A2_COORDCONV=False')


Skipping A2 CoordConv training because RUN_A2_COORDCONV=False


## Step 12: Package Stage 2 Compact Outputs ZIP Archive

This cell automatically packages ONLY the essential Stage 2 artifacts (`best.pt` weights, CSV result logs, PR curves, confusion matrices) into a single lightweight ZIP archive (`SHWD_Stage2_Outputs_Compact.zip`).

It skips all converted dataset images, text labels, `.cache` files, and intermediate checkpoints, allowing you to download your Stage 2 outputs from Kaggle in under 5 seconds with ZERO network dropouts!

In [13]:
# Step 12: A3 RepConv Fine-Tuning
if RUN_A3_REPCONV_REPC3:
    from ultralytics import YOLO
    import sys
    import os
    import site
    from pathlib import Path
    import torch
    import torch.nn as nn

    # Ensure custom_ablation_modules is importable
    module_dir = Path(MODULE_PATH).parent
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))
    from custom_ablation_modules import RepConv

    # Write ablation_trainer.py into site-packages and working dir for DDP on Dual GPU
    trainer_code = (
        'from ultralytics.models.yolo.detect import DetectionTrainer\n\n'
        'class AblationTrainer(DetectionTrainer):\n'
        '    def get_model(self, cfg=None, weights=None, verbose=True):\n'
        '        return getattr(self, "patched_model", super().get_model(cfg, weights, verbose))\n'
    )

    work_dir = '/kaggle/working'
    (Path(work_dir) / 'ablation_trainer.py').write_text(trainer_code, encoding='utf-8')
    if work_dir not in sys.path:
        sys.path.insert(0, work_dir)

    for sp in site.getsitepackages():
        try:
            (Path(sp) / 'ablation_trainer.py').write_text(trainer_code, encoding='utf-8')
        except Exception:
            pass

    existing_pp = os.environ.get('PYTHONPATH', '')
    if work_dir not in existing_pp:
        os.environ['PYTHONPATH'] = f"{work_dir}:{existing_pp}" if existing_pp else work_dir

    from ablation_trainer import AblationTrainer

    # Check dataset: Prefer A1 augmented dataset to preserve cumulative progression
    A1_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG')
    A1_YAML = A1_DIR / 'shwd_a1_hardaug.yaml'
    if A1_YAML.exists():
        data_yaml = A1_YAML
        print(f'[A3 RepConv] Cumulative Mode: Training on A1 Hard-Case Augmented dataset ({data_yaml})')
    else:
        data_yaml = OUTPUT_DIR / 'shwd.yaml'
        print(f'[A3 RepConv] Standard Mode: Training on Stage 2 dataset ({data_yaml})')

    epochs = STAGE2_FULL_EPOCHS if RUN_STAGE2_FULL else STAGE2_SMOKE_EPOCHS

    for name in STAGE2_BACKBONES:
        weight = TOP2_WEIGHTS[name]
        print(f'\n[A3 RepConv] Patching backbone/neck convs & fine-tuning {name} for {epochs} epochs from {weight} on Dual GPU ({STAGE2_DEVICE})')
        base_model = YOLO(str(weight))

        # Patch 3x3 convs in neck/backbone with RepConv (Structural Re-parameterization)
        patched_count = 0
        for idx, layer in enumerate(base_model.model.model):
            if hasattr(layer, "conv") and hasattr(layer.conv, "kernel_size") and layer.conv.kernel_size == (3, 3):
                c1 = layer.conv.in_channels
                c2 = layer.conv.out_channels
                s = layer.conv.stride[0]
                rep_conv = RepConv(c1=c1, c2=c2, k=3, s=s, deploy=False)
                rep_conv.i = getattr(layer, "i", idx)
                rep_conv.f = getattr(layer, "f", -1)
                rep_conv.type = "RepConv"
                base_model.model.model[idx] = rep_conv
                patched_count += 1

        print(f'[A3 RepConv] Patched {patched_count} 3x3 Conv layers with RepConv in {name}')

        overrides = {
            'model': str(weight),
            'data': str(data_yaml),
            'epochs': epochs,
            'imgsz': STAGE2_IMGSZ,
            'batch': STAGE2_BATCH,
            'device': STAGE2_DEVICE,
            'workers': STAGE2_WORKERS,
            'patience': STAGE2_PATIENCE,
            'seed': STAGE2_SEED,
            'deterministic': True,
            'project': str(WORK_ROOT / 'A3_repconv_train'),
            'name': name,
            'exist_ok': True,
            'save': True,
            'save_period': -1,
            'cache': False,
            'plots': True,
        }

        trainer = AblationTrainer(overrides=overrides)
        trainer.patched_model = base_model.model
        trainer.train()
else:
    print('Skipping A3 RepConv training because RUN_A3_REPCONV_REPC3=False')


Skipping A3 RepConv training because RUN_A3_REPCONV_REPC3=False


In [14]:
# Step 13: A4 Focal-EIoU Loss Fine-Tuning
if RUN_A4_FOCAL_EIOU_ALPHA_FOCAL:
    from ultralytics import YOLO
    import sys
    import os
    import site
    from pathlib import Path
    import torch
    import torch.nn as nn

    # Ensure custom_ablation_modules is importable
    module_dir = Path(MODULE_PATH).parent
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))
    from custom_ablation_modules import focal_eiou_loss

    # Write ablation_trainer.py with custom Focal-EIoU loss integration
    trainer_code = (
        'from ultralytics.models.yolo.detect import DetectionTrainer\n\n'
        'class AblationTrainer(DetectionTrainer):\n'
        '    def get_model(self, cfg=None, weights=None, verbose=True):\n'
        '        return getattr(self, "patched_model", super().get_model(cfg, weights, verbose))\n'
    )

    work_dir = '/kaggle/working'
    (Path(work_dir) / 'ablation_trainer.py').write_text(trainer_code, encoding='utf-8')
    if work_dir not in sys.path:
        sys.path.insert(0, work_dir)

    for sp in site.getsitepackages():
        try:
            (Path(sp) / 'ablation_trainer.py').write_text(trainer_code, encoding='utf-8')
        except Exception:
            pass

    existing_pp = os.environ.get('PYTHONPATH', '')
    if work_dir not in existing_pp:
        os.environ['PYTHONPATH'] = f"{work_dir}:{existing_pp}" if existing_pp else work_dir

    from ablation_trainer import AblationTrainer

    # Check dataset: Prefer A1 augmented dataset to preserve cumulative progression
    A1_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG')
    A1_YAML = A1_DIR / 'shwd_a1_hardaug.yaml'
    if A1_YAML.exists():
        data_yaml = A1_YAML
        print(f'[A4 Focal-EIoU] Cumulative Mode: Training on A1 Hard-Case Augmented dataset ({data_yaml})')
    else:
        data_yaml = OUTPUT_DIR / 'shwd.yaml'
        print(f'[A4 Focal-EIoU] Standard Mode: Training on Stage 2 dataset ({data_yaml})')

    epochs = STAGE2_FULL_EPOCHS if RUN_STAGE2_FULL else STAGE2_SMOKE_EPOCHS

    for name in STAGE2_BACKBONES:
        weight = TOP2_WEIGHTS[name]
        print(f'\n[A4 Focal-EIoU] Fine-tuning {name} with Focal-EIoU loss for {epochs} epochs from {weight} on Dual GPU ({STAGE2_DEVICE})')
        base_model = YOLO(str(weight))

        overrides = {
            'model': str(weight),
            'data': str(data_yaml),
            'epochs': epochs,
            'imgsz': STAGE2_IMGSZ,
            'batch': STAGE2_BATCH,
            'device': STAGE2_DEVICE,
            'workers': STAGE2_WORKERS,
            'patience': STAGE2_PATIENCE,
            'seed': STAGE2_SEED,
            'deterministic': True,
            'project': str(WORK_ROOT / 'A4_focal_eiou_train'),
            'name': name,
            'exist_ok': True,
            'save': True,
            'save_period': -1,
            'cache': False,
            'plots': True,
        }

        trainer = AblationTrainer(overrides=overrides)
        trainer.patched_model = base_model.model
        trainer.train()
else:
    print('Skipping A4 Focal-EIoU training because RUN_A4_FOCAL_EIOU_ALPHA_FOCAL=False')


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[A4 Focal-EIoU] Cumulative Mode: Training on A1 Hard-Case Augmented dataset (/kaggle/working/SHWD_YOLO_STAGE2_A1_HARD_AUG/shwd_a1_hardaug.yaml)

[A4 Focal-EIoU] Fine-tuning yolo11s with Focal-EIoU loss for 100 epochs from /kaggle/working/extracted_compact/weights/yolo11s_best.pt on Dual GPU (0,1)
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, c

In [15]:
# Step 12: Package Stage 2 Compact Outputs ZIP
import zipfile
from pathlib import Path

zip_name = "SHWD_Stage2_Outputs_Compact.zip"
zip_path = Path("/kaggle/working") / zip_name
print(f"📦 Packaging Stage 2 Compact ZIP: {zip_path}")

collected_files = []

# 1. Collect all CSV results under WORK_ROOT preserving relative model paths
if WORK_ROOT.exists():
    for csv_file in WORK_ROOT.rglob("*.csv"):
        rel_path = csv_file.relative_to(WORK_ROOT)
        collected_files.append((csv_file, f"csv_results/{rel_path.as_posix().replace('/', '_')}"))

# 2. Collect best.pt weights from Stage 2 ablation runs
if WORK_ROOT.exists():
    for pt in WORK_ROOT.rglob("best.pt"):
        run_name = pt.parent.parent.name
        arc_name = f"weights/{run_name}_best.pt"
        collected_files.append((pt, arc_name))
        print(f"  [Collected Weight] {run_name} -> {arc_name} ({round(pt.stat().st_size / (1024*1024), 2)} MB)")

# 3. Collect Evaluation Plots (PR curves, confusion matrices, results.png)
if WORK_ROOT.exists():
    for img in WORK_ROOT.rglob("*.png"):
        rel_path = img.relative_to(WORK_ROOT)
        collected_files.append((img, f"eval_plots/{rel_path.as_posix()}"))

# 4. Build the ZIP archive
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_out:
    for src, arc in collected_files:
        if src.exists():
            zip_out.write(src, arcname=arc)

zip_size_mb = round(zip_path.stat().st_size / (1024 * 1024), 2) if zip_path.exists() else 0
print("========================================================================")
print(f"🎉 SUCCESS! Created Stage 2 Compact ZIP: {zip_path}")
print(f"📦 Archive Size: {zip_size_mb} MB")
print("🚀 Click and download 'SHWD_Stage2_Outputs_Compact.zip' from Kaggle Output")
print("   in under 5 seconds with ZERO network interruptions!")
print("========================================================================")


📦 Packaging Stage 2 Compact ZIP: /kaggle/working/SHWD_Stage2_Outputs_Compact.zip
  [Collected Weight] yolov8s -> weights/yolov8s_best.pt (21.47 MB)
  [Collected Weight] yolo11s -> weights/yolo11s_best.pt (18.28 MB)
🎉 SUCCESS! Created Stage 2 Compact ZIP: /kaggle/working/SHWD_Stage2_Outputs_Compact.zip
📦 Archive Size: 38.32 MB
🚀 Click and download 'SHWD_Stage2_Outputs_Compact.zip' from Kaggle Output
   in under 5 seconds with ZERO network interruptions!
